# Lab 12 - Evaluate the live Azure AI Search RAG path

## What are we evaluating?

Lab 8 evaluated fixed examples. This lab runs the complete live path against the WHO Search index:

```text
question -> Azure AI Search -> selected context -> cited model answer
```

A failure can occur at any stage, so we keep the evidence separate:

| Stage | What we check |
|---|---|
| Retrieval | Did Search return the expected publication, and how highly was it ranked? |
| Context | Do the selected chunks contain the required facts? |
| Generation | Did the answer cover every required part? |
| Citations | Does each claim cite a source actually supplied to the model? |
| Semantic quality | Do repeated judges find the answer grounded and complete? |
| Operations | How long did each stage take, and what usage did it create? |

This separation tells you where to fix a failure. If the context contains the facts but the answer omits them, investigate generation. If the facts never reach the context, investigate retrieval or context selection.

The benchmark contains three WHO topics. A release gate combines exact checks with repeated model-based scores.

## New words

- **Recall@k** - how much expected evidence appears in the first `k` results.
- **Precision@k** - how much of those first `k` results is expected evidence.
- **Reciprocal rank** - rewards finding the first expected result early: rank 1 scores 1, rank 2 scores 0.5.
- **Generation context** - the exact retrieved text sent to the answer model.
- **Answer contract** - the required answer parts and citation rules declared before generation.

## Before you start

Complete Lab 3 and set the Foundry project, model deployment, Search endpoint and index name. You need Foundry User and Search Index Data Reader access.

Replace each `...` blank before running its cell.

> Three topics are workshop smoke coverage, not clinical validation or a production reliability target.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0" "azure-search-documents==12.0.0"

## 0. Connect and load the evaluation tools

The next cell prepares two Azure clients:

- `SearchClient` runs live queries against the WHO index.
- `AIProjectClient` provides the model used to generate answers and the Foundry evaluators used later.

It also imports `evaluation_utils.py`. That helper contains deterministic calculations shared by Labs 12 and 13: retrieval metrics, context checks, citation validation, token counts and optional cost estimates. Keeping these rules in one module ensures both RAG paths are measured the same way.

The answer model never receives the benchmark's expected answer or evidence labels. Those remain evaluation-only data.

`JUDGE_REPEATS` controls how many times each frozen answer is scored. An odd count allows the notebook to use a median without a tie between two middle values.

**You should see** the Search index, semantic configuration, generator model, evaluator model and judge repeat count.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
from statistics import median
from uuid import uuid4

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper_path = folder / "labs" / "day_2" / "evaluation_utils.py"
    if helper_path.exists():
        REPO_ROOT = folder
        sys.path.insert(0, str(helper_path.parent))
        break
else:
    raise FileNotFoundError("labs/day_2/evaluation_utils.py was not found.")

from evaluation_utils import (
    answer_json_schema,
    context_signal_coverage,
    document_evidence_keys,
    estimate_cost_usd,
    format_search_context,
    load_cases,
    price_rates_from_env,
    primitive,
    ranked_evidence_metrics,
    render_cited_answer,
    response_token_usage,
    retrieval_recall,
    validate_cited_answer,
)
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from azure.search.documents import SearchClient

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
EVALUATOR_MODEL = os.getenv("RAG_EVALUATOR_MODEL", "") or MODEL_DEPLOYMENT
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX_NAME", "")
SEMANTIC_CONFIGURATION = os.getenv(
    "AZURE_SEARCH_SEMANTIC_CONFIGURATION", "who-guidelines-semantic"
)
JUDGE_REPEATS = int(os.getenv("RAG_EVAL_JUDGE_REPEATS", "3"))
if not 3 <= JUDGE_REPEATS <= 9 or JUDGE_REPEATS % 2 == 0:
    raise ValueError("RAG_EVAL_JUDGE_REPEATS must be an odd number from 3 to 9.")
missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
        "AZURE_SEARCH_ENDPOINT": SEARCH_ENDPOINT,
        "AZURE_SEARCH_INDEX_NAME": SEARCH_INDEX,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


def wait_for_run(eval_id, run, expected_items):
    deadline = time.monotonic() + 1200
    while run.status not in ("completed", "failed", "canceled"):
        if time.monotonic() > deadline:
            raise TimeoutError("Evaluation exceeded 20 minutes.")
        time.sleep(5)
        run = client.evals.runs.retrieve(run_id=run.id, eval_id=eval_id)
        print("status:", run.status)
    items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    if run.status != "completed":
        raise RuntimeError(f"Evaluation ended as {run.status}: {primitive(getattr(run, 'error', None))}")
    item_deadline = time.monotonic() + 120
    while len(items) < expected_items:
        if time.monotonic() > item_deadline:
            raise TimeoutError(f"Only {len(items)}/{expected_items} output items became visible.")
        time.sleep(2)
        items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id))
    return run, items


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, credential=credential)
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=300, max_retries=0)
RATES = price_rates_from_env()
print({
    "index": SEARCH_INDEX,
    "semantic_configuration": SEMANTIC_CONFIGURATION,
    "generator": MODEL_DEPLOYMENT,
    "judge": EVALUATOR_MODEL,
    "judge_repeats": JUDGE_REPEATS,
})

## 1. Load the versioned benchmark

The benchmark contains three questions matching the workshop's WHO topics: IPC staffing, hypertension treatment and type 2 diabetes diagnosis.

Each case declares the expected evidence before live Search runs:

- `expected_evidence_groups`: stable publication identifiers that should be retrieved;
- `required_context_signals`: words or phrases showing that the selected chunks contain needed facts;
- `required_answer_parts`: the pieces a complete response must cover;
- `ground_truth`: an independently written reference used only during evaluation.

These fields serve different purposes. Finding the expected publication does not guarantee that the selected chunks contain every required fact, so the lab checks both publication retrieval and context signals.

The benchmark file is versioned so later runs use the same expectations. None of its expected answers or signals are included in the generation prompt.

**You should see** the dataset version, answer-contract version and three Search case IDs.

In [ ]:
DATASET_PATH = REPO_ROOT / "labs" / "data" / "medical_rag_evaluation_cases.json"
DATASET, SEARCH_CASES = load_cases(DATASET_PATH, "azure_ai_search")
assert DATASET["dataset_id"] == "umc-who-rag-evaluation-v1"
assert len(SEARCH_CASES) == 3
print({
    "dataset": DATASET["dataset_id"],
    "answer_contract": DATASET["answer_contract_version"],
    "cases": [case["case_id"] for case in SEARCH_CASES],
})

### To-Do 1 - Choose retrieval depths

Search returns a ranked list. We use two depths for different jobs:

- `TOP_K = 5` defines the first five results used for headline retrieval metrics. These ranks remain untouched so weak top results stay visible.
- `CANDIDATE_CAP = 10` allows context selection to inspect up to ten results, giving generation a bounded chance to recover additional useful chunks.

A deeper candidate set can improve recall, but it also examines more text and may introduce less relevant evidence. It must remain bounded rather than retrieving the whole index.

Set the two values below.

**Predict:** if Recall@5 is low but candidate recall at 10 is high, what did the additional five results recover, and what extra noise might they add?

**Key concept:** measure the original ranking separately from the larger pool available for generation.

<details><summary>Show solution code</summary>

```python
TOP_K = 5
CANDIDATE_CAP = 10
```

</details>

In [ ]:
TOP_K = ...  # TODO 1: rank depth used for headline retrieval quality.
CANDIDATE_CAP = ...  # TODO 1: bounded depth available for coherent generation context.
check_todos(TOP_K=TOP_K, CANDIDATE_CAP=CANDIDATE_CAP)
assert TOP_K == 5
assert CANDIDATE_CAP == 10
print({"top_k": TOP_K, "candidate_cap": CANDIDATE_CAP})

## 2. Run the live Search-to-answer path

The next two code cells execute the full path for each benchmark case.

1. Azure AI Search returns up to ten semantically ranked chunks.
2. The first five are measured for recall, precision and rank.
3. The top result identifies the primary `publication_id`. Only candidate chunks from that same WHO publication enter the generation context, which avoids blending similarly worded facts from different guidelines.
4. The context is checked for the benchmark's required evidence signals.
5. Each context chunk receives a local source label such as `[S1]`.
6. The model returns strict JSON with one or more claims for each required answer part and cites only those supplied source labels.
7. The helper validates answer-part coverage and citations, then records latency, token use and Search request count.

The first code cell defines the cited-answer generator. The second performs live retrieval, calls that generator and stores one diagnostic row per benchmark case before any model-based evaluator scores it.

**You should see** three diagnostic records and three cited answers, followed by a `PASS` message.

In [ ]:
def generate_cited_answer(case, context, valid_source_ids):
    required_parts = case["required_answer_parts"]
    part_ids = [part["id"] for part in required_parts]
    parts_text = "\n".join(
        f"- {part['id']}: {part['description']}" for part in required_parts
    )
    schema = answer_json_schema(part_ids, valid_source_ids)
    response = client.responses.create(
        model=MODEL_DEPLOYMENT,
        instructions=(
            "Answer only from the supplied WHO Search sources. Cover each required answer part "
            "with one or more concise factual claims. Give every claim exactly one answer_part_id "
            "and one or more supporting source_ids. Put a part in insufficient_evidence only when "
            "the supplied sources cannot support it, and never both answer and mark the same part "
            "insufficient. Do not add patient-specific advice or use source IDs outside the schema."
        ),
        input=(
            f"Question: {case['query']}\n\n"
            f"Required answer parts ({DATASET['answer_contract_version']}):\n{parts_text}\n\n"
            f"Search sources:\n{context}"
        ),
        text={
            "format": {
                "type": "json_schema",
                "name": "medical_search_answer",
                "strict": True,
                "schema": schema,
            }
        },
        max_output_tokens=1400,
    )
    validation = validate_cited_answer(
        json.loads(response.output_text),
        valid_source_ids,
        part_ids,
    )
    return response, validation, render_cited_answer(validation)


print("Cited-answer generator ready.")

In [ ]:
SEARCH_ROWS = []
for case in SEARCH_CASES:
    total_started = time.perf_counter()
    retrieval_started = time.perf_counter()
    documents = [
        dict(result)
        for result in search_client.search(
            search_text=case["query"],
            query_type="semantic",
            semantic_configuration_name=SEMANTIC_CONFIGURATION,
            top=CANDIDATE_CAP,
            select=[
                "chunk_id",
                "parent_id",
                "chunk",
                "document_title",
                "source_url",
                "publication_id",
                "topic",
                "metadata_storage_name",
            ],
        )
    ]
    retrieval_ms = (time.perf_counter() - retrieval_started) * 1000
    if not documents:
        raise RuntimeError(f"{case['case_id']}: Search returned no documents.")

    top_documents = documents[:TOP_K]
    top_keys = set().union(*(document_evidence_keys(doc) for doc in top_documents))
    candidate_keys = set().union(*(document_evidence_keys(doc) for doc in documents))
    recall_at_k = retrieval_recall(case["expected_evidence_groups"], top_keys)
    candidate_recall = retrieval_recall(case["expected_evidence_groups"], candidate_keys)
    ranking = ranked_evidence_metrics(case["expected_evidence_groups"], top_documents)

    primary_publication = str(documents[0].get("publication_id") or "").strip()
    if not primary_publication:
        raise RuntimeError(f"{case['case_id']}: the top result has no publication_id.")
    generation_documents = [
        document
        for document in documents
        if str(document.get("publication_id") or "").strip() == primary_publication
    ]
    context = format_search_context(generation_documents)
    signal_result = context_signal_coverage(case["required_context_signals"], context)
    generation_keys = set().union(
        *(document_evidence_keys(doc) for doc in generation_documents)
    )
    generation_document_recall = retrieval_recall(
        case["expected_evidence_groups"], generation_keys
    )
    valid_source_ids = [f"S{index}" for index in range(1, len(generation_documents) + 1)]

    generation_started = time.perf_counter()
    response, citations, answer = generate_cited_answer(case, context, valid_source_ids)
    generation_ms = (time.perf_counter() - generation_started) * 1000
    usage = response_token_usage(response, semantic_requests=1)
    cost = estimate_cost_usd(usage, RATES)

    row = {
        "case_id": case["case_id"],
        "query": case["query"],
        "ground_truth": case["ground_truth"],
        "context": context,
        "response": answer,
        "raw_sources": [
            {
                "rank": rank,
                "publication_id": document.get("publication_id"),
                "chunk_id": document.get("chunk_id"),
                "reranker_score": document.get("@search.reranker_score"),
            }
            for rank, document in enumerate(documents, start=1)
        ],
        "primary_publication": primary_publication,
        "retrieval_recall_at_5": recall_at_k["recall"],
        "candidate_recall_at_10": candidate_recall["recall"],
        "precision_at_5": ranking["precision"],
        "top_1_match": ranking["top_1_match"],
        "first_match_rank": ranking["first_match_rank"],
        "reciprocal_rank": ranking["reciprocal_rank"],
        "generation_document_recall": generation_document_recall["recall"],
        "context_signal_coverage": signal_result["coverage"],
        "context_signal_details": signal_result["signals"],
        "answer_part_coverage": citations["answer_part_coverage"],
        "insufficient_evidence": citations["insufficient_evidence"],
        "citation_coverage": citations["citation_coverage"],
        "citation_validity": citations["citation_validity"],
        "retrieval_ms": round(retrieval_ms, 2),
        "generation_ms": round(generation_ms, 2),
        "total_ms": round((time.perf_counter() - total_started) * 1000, 2),
        **usage,
        **cost,
    }
    SEARCH_ROWS.append(row)
    print(json.dumps({
        key: row[key]
        for key in (
            "case_id",
            "primary_publication",
            "raw_sources",
            "retrieval_recall_at_5",
            "candidate_recall_at_10",
            "precision_at_5",
            "top_1_match",
            "reciprocal_rank",
            "generation_document_recall",
            "context_signal_coverage",
            "answer_part_coverage",
            "citation_coverage",
            "citation_validity",
            "retrieval_ms",
            "generation_ms",
            "model_input_tokens",
            "model_output_tokens",
            "estimated_cost_usd",
        )
    }, indent=2))
    print(answer, "\n")

assert len(SEARCH_ROWS) == len(SEARCH_CASES)
print("PASS - every benchmark case completed live retrieval and strictly cited generation.")

## 3. Judge each frozen answer repeatedly

The exact checks above verify retrieval, required parts and citation IDs. Foundry now applies two model-based evaluators to each completed row:

- **Groundedness** compares the response with the exact context used to generate it.
- **Response Completeness** compares the response with the independent expected answer.

The response is frozen before judging, so repeated scores measure evaluator variability rather than generating new answers until one passes.

Each row is submitted `JUDGE_REPEATS` times. The notebook keeps every score and reason, then uses the median as the predeclared consensus. With the default of three cases and three repeats, Foundry scores nine items.

Evaluator token usage is reported separately from the tokens used by the live RAG application path.

**You should see** nine output items, all raw scores and reasons, median scores for each case and a Foundry report URL.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

eval_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "eval_item_id": {"type": "string"},
            "run_case_id": {"type": "string"},
            "judge_repeat": {"type": "integer"},
            "query": {"type": "string"},
            "context": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": [
            "eval_item_id",
            "run_case_id",
            "judge_repeat",
            "query",
            "context",
            "response",
            "ground_truth",
        ],
        "additionalProperties": False,
    },
    include_sample_schema=True,
)
criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "query": "{{item.query}}",
            "context": "{{item.context}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="response_completeness",
        evaluator_name="builtin.response_completeness",
        initialization_parameters={"deployment_name": EVALUATOR_MODEL},
        data_mapping={
            "ground_truth": "{{item.ground_truth}}",
            "response": "{{item.response}}",
        },
    ),
]
evaluation = client.evals.create(
    name=f"day2-search-rag-eval-{SUFFIX}",
    data_source_config=eval_config,
    testing_criteria=criteria,
)
eval_items = [
    {
        "item": {
            "eval_item_id": f"{row['case_id']}-J{repeat}",
            "run_case_id": row["case_id"],
            "judge_repeat": repeat,
            **{key: row[key] for key in ("query", "context", "response", "ground_truth")},
        }
    }
    for row in SEARCH_ROWS
    for repeat in range(1, JUDGE_REPEATS + 1)
]
eval_run = client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"day2-search-rag-run-{SUFFIX}",
    metadata={
        "dataset": DATASET["dataset_id"],
        "answer_contract": DATASET["answer_contract_version"],
        "target": "azure-ai-search",
        "judge_repeats": str(JUDGE_REPEATS),
    },
    data_source={
        "type": "jsonl",
        "source": {"type": "file_content", "content": eval_items},
    },
)
print({"evaluation_id": evaluation.id, "run_id": eval_run.id, "items": len(eval_items)})

In [ ]:
eval_run, output_items = wait_for_run(evaluation.id, eval_run, len(eval_items))
METRICS = ("groundedness", "response_completeness")
scores = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in SEARCH_ROWS
}
reasons = {
    row["case_id"]: {metric: {} for metric in METRICS}
    for row in SEARCH_ROWS
}
for item in output_items:
    data = primitive(item)
    source = data["datasource_item"]
    case_id = source["run_case_id"]
    repeat = int(source["judge_repeat"])
    for metric in METRICS:
        result = next(result for result in data["results"] if result["name"] == metric)
        if result.get("error") or result.get("status") in ("failed", "error", "canceled"):
            raise RuntimeError(f"{case_id}-J{repeat} {metric} failed: {result}")
        reason = str(result.get("reason") or "").strip()
        if not reason:
            raise RuntimeError(f"{case_id}-J{repeat} {metric} returned no reason.")
        scores[case_id][metric][repeat] = float(result["score"])
        reasons[case_id][metric][repeat] = reason

for row in SEARCH_ROWS:
    for metric in METRICS:
        samples = [scores[row["case_id"]][metric][repeat] for repeat in range(1, JUDGE_REPEATS + 1)]
        row[f"{metric}_samples"] = samples
        row[f"{metric}_reasons"] = [
            reasons[row["case_id"]][metric][repeat]
            for repeat in range(1, JUDGE_REPEATS + 1)
        ]
        row[f"{metric}_mean"] = round(sum(samples) / len(samples), 3)
        row[metric] = float(median(samples))

print({
    "consensus": {
        row["case_id"]: {
            metric: {
                "samples": row[f"{metric}_samples"],
                "median": row[metric],
                "reasons": row[f"{metric}_reasons"],
            }
            for metric in METRICS
        }
        for row in SEARCH_ROWS
    },
    "evaluator_usage": [primitive(item) for item in (getattr(eval_run, "per_model_usage", None) or [])],
    "report_url": getattr(eval_run, "report_url", None),
})

### To-Do 2 - Set the release thresholds

Choose the quality policy before inspecting the final decisions. Set both median model-based thresholds to `4.0` on the 1-to-5 scale.

A case ships only when two groups of checks pass:

**Deterministic checks**

- the expected publication is first and fully present in the candidate and generation sets;
- all required context signals and answer parts are covered;
- no required part is marked unsupported;
- every claim has a valid supplied citation.

**Semantic checks**

- median Groundedness is at least 4;
- median Response Completeness is at least 4.

A strong average cannot compensate for a broken citation or missing required fact. These thresholds are workshop defaults, not universal clinical standards.

**Key concept:** combine exact contracts with semantic judgement; do not reduce the release decision to one score.

<details><summary>Show solution code</summary>

```python
MIN_GROUNDEDNESS = 4.0
MIN_COMPLETENESS = 4.0
```

</details>

In [ ]:
MIN_GROUNDEDNESS = ...  # TODO 2: median groundedness threshold on the 1-to-5 scale.
MIN_COMPLETENESS = ...  # TODO 2: median completeness threshold on the 1-to-5 scale.
check_todos(MIN_GROUNDEDNESS=MIN_GROUNDEDNESS, MIN_COMPLETENESS=MIN_COMPLETENESS)
assert MIN_GROUNDEDNESS == MIN_COMPLETENESS == 4.0

for row in SEARCH_ROWS:
    row["deterministic_gate_passed"] = all([
        row["candidate_recall_at_10"] == 1.0,
        row["generation_document_recall"] == 1.0,
        row["context_signal_coverage"] == 1.0,
        row["top_1_match"],
        row["answer_part_coverage"] == 1.0,
        not row["insufficient_evidence"],
        row["citation_coverage"] == 1.0,
        row["citation_validity"] == 1.0,
    ])
    row["semantic_gate_passed"] = (
        row["groundedness"] >= MIN_GROUNDEDNESS
        and row["response_completeness"] >= MIN_COMPLETENESS
    )
    row["release_gate_passed"] = (
        row["deterministic_gate_passed"] and row["semantic_gate_passed"]
    )
    row["decision"] = "ship" if row["release_gate_passed"] else "mitigate"
    print(json.dumps({
        "case_id": row["case_id"],
        "decision": row["decision"],
        "retrieval": {
            "recall_at_5": row["retrieval_recall_at_5"],
            "candidate_recall_at_10": row["candidate_recall_at_10"],
            "precision_at_5": row["precision_at_5"],
            "top_1_match": row["top_1_match"],
            "reciprocal_rank": row["reciprocal_rank"],
        },
        "context_signal_coverage": row["context_signal_coverage"],
        "answer_part_coverage": row["answer_part_coverage"],
        "citation_coverage": row["citation_coverage"],
        "citation_validity": row["citation_validity"],
        "groundedness": {
            "samples": row["groundedness_samples"],
            "median": row["groundedness"],
            "reasons": row["groundedness_reasons"],
        },
        "response_completeness": {
            "samples": row["response_completeness_samples"],
            "median": row["response_completeness"],
            "reasons": row["response_completeness_reasons"],
        },
        "latency_ms": {
            "retrieval": row["retrieval_ms"],
            "generation": row["generation_ms"],
            "total": row["total_ms"],
        },
        "usage": {
            "model_input_tokens": row["model_input_tokens"],
            "model_output_tokens": row["model_output_tokens"],
            "semantic_requests": row["semantic_requests"],
        },
        "estimated_cost_usd": row["estimated_cost_usd"],
    }, indent=2))

## Verify that the evidence pipeline completed

A useful evaluation can legitimately recommend `mitigate`. The final check therefore does not force every live row to ship.

Instead, it verifies that all three cases completed and produced the evidence needed for a decision: retrieval metrics, citation validation, repeated judge scores and reasons, latency, usage and a `ship` or `mitigate` outcome.

If a row recommends mitigation, diagnose the first failing stage before changing prompts, Search settings or thresholds.

**You should see** a `PASS` message and one decision per benchmark case.

In [ ]:
assert eval_run.status == "completed"
assert len(SEARCH_ROWS) == len(SEARCH_CASES) == 3
assert len(output_items) == len(SEARCH_ROWS) * JUDGE_REPEATS
assert all(0.0 <= row["retrieval_recall_at_5"] <= 1.0 for row in SEARCH_ROWS)
assert all(0.0 <= row["precision_at_5"] <= 1.0 for row in SEARCH_ROWS)
assert all(0.0 <= row["citation_validity"] <= 1.0 for row in SEARCH_ROWS)
assert all(len(row["groundedness_samples"]) == JUDGE_REPEATS for row in SEARCH_ROWS)
assert all(len(row["response_completeness_reasons"]) == JUDGE_REPEATS for row in SEARCH_ROWS)
assert all(row["retrieval_ms"] > 0 and row["total_ms"] >= row["retrieval_ms"] for row in SEARCH_ROWS)
assert all(row["decision"] in {"ship", "mitigate"} for row in SEARCH_ROWS)
print("PASS - retrieval, context, generation, citation, judge, latency, usage and release evidence are populated.")
print("Decisions:", {row["case_id"]: row["decision"] for row in SEARCH_ROWS})

## Diagnose the first failing stage

Use the earliest failed check to decide where to investigate:

| First failure | Likely area |
|---|---|
| Expected publication missing or ranked low | Search query or ranking |
| Publication found but required context signals missing | Chunk selection, candidate depth or source content |
| Context complete but answer parts missing | Generation instructions or answer contract |
| Citation missing or unknown | Structured output or source-label handling |
| Exact checks pass but judge score is low | Read the judge reason and compare response, context and expected answer |
| Quality passes but latency or usage is high | Operational optimization |

Do not lower a quality threshold merely to make a live result pass. First identify whether the failure belongs to retrieval, context, generation, citation handling or model-based judging.

<details><summary>Optional cost estimates</summary>

The notebook always reports token counts and Search requests. It estimates USD only when you provide the applicable rate variables from your own price sheet. Estimates are not Azure billing records.

</details>

## What you learned

- A live RAG evaluation keeps retrieval, context, generation, citations, semantic quality and operations separate.
- Recall and rank describe whether expected evidence appeared; context signals describe whether the selected text contains required facts.
- Restricting generation context to one publication avoids mixing guideline domains.
- Required answer parts expose polished but incomplete responses.
- Citation coverage and validity are exact checks; Groundedness and Completeness add model-based judgement.
- Repeating a frozen answer reveals judge variability without retrying generation until it passes.
- Release decisions should include latency and usage as well as quality.

**Check your understanding**

1. Candidate recall is 1.0 but context-signal coverage is 0.5. What failed?
2. Every citation ID is valid but Groundedness is low. What does citation validity not prove?
3. Why are three benchmark cases not a production reliability target?

<details><summary>Compare your answers</summary>

1. The expected publication was found, but the selected chunks did not contain every required fact.
2. It proves the referenced source was supplied, not that the claim accurately represents that source.
3. Three cases do not represent real query frequency, languages, edge cases or quality variation.

</details>

Further reading: [semantic ranking](https://learn.microsoft.com/azure/search/semantic-how-to-query-request), [RAG evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rag-evaluators), and [RAG evaluation design](https://learn.microsoft.com/azure/architecture/ai-ml/guide/rag/rag-solution-design-and-evaluation-guide).

**Expected artifact:** three row-level reports containing retrieval, context, citation, judge, latency, usage and release-decision evidence.

**Finish:** the final cell closes local clients. The Foundry evaluation report remains available.

**Next:** Lab 13 runs the matching benchmark through Foundry IQ and measures repeatability and retrieval activity.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed local clients. The evaluation report remains in Foundry.")